# 센다 실험 — 단계 A, **A8** (A5n 설정 + 재시작) 시드 1~8 (코랩 GPU)
**A5n과 같은 설정에 "재시작"만 추가했습니다.** 학습 도중 두 번 검증하고(전체의 40%, 70% 지점), 검증 점수(n 대 n+1 균형 정확도)가 0.95 미만이면 **눈은 그대로 두고 화자의 기호 선택부분과 판정자의 쪽지 읽는 부분만 처음부터 다시 시작**합니다(최대 2번). 끝날 때 검증 점수가 가장 좋았던 상태를 최종 모델로 고릅니다(고르는 기준은 같다/다르다 결과뿐이고 개수 정답은 쓰지 않음).

⚠ **지표 수정:** 전에는 n 대 n+1 정확도를 "다른 쌍"만으로 쟀습니다(늘 "다르다"라고 답해도 만점). 이제 같은 쌍과 다른 쌍을 반반으로 잽니다. 옛 결과(A5n 등)의 "n대n+1 ≥90%" 숫자는 부풀려져 있었습니다.

**처음 한 번만:** 런타임 → 런타임 유형 변경 → GPU(T4). 위에서부터 셀을 차례로 실행.
**끊겼을 때:** 같은 셀들을 처음부터 다시 실행하면 드라이브 체크포인트에서 이어갑니다. 끝난 시드는 건너뜁니다.
결과 폴더는 `countworld_runs2`(옛 결과와 섞이지 않음).

## 1. GPU 확인과 구글 드라이브 연결

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/countworld_runs2'
os.makedirs(BASE, exist_ok=True); print(BASE)

## 2. 저장소 내려받기 (최신 코드로)

In [ ]:
import os, subprocess
REPO = 'gitdd12/integral_discovery'
BRANCH = 'ccr-206f948d-nkcq34'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    token = None
    url = f'https://github.com/{REPO}.git'
if not os.path.exists('/content/integral_discovery'):
    subprocess.run(['git','clone','-b',BRANCH,url,'/content/integral_discovery'], check=True)
else:
    subprocess.run(['git','-C','/content/integral_discovery','pull'], check=True)
%cd /content/integral_discovery
!git log --oneline | head -3

## 3. 설정 (바꾸지 마세요. 시드만 줄이거나 늘릴 수 있습니다)

In [ ]:
SEEDS = list(range(1, 9))    # 시드 1~8
STEPS = 12000
MAX_PARALLEL = 3
EXPERIMENTS = {
    'A8': dict(USAGE='1.0', MI='1.0', RAMP='0.3', ENT='0.03', FOCUS='2',
               RESTARTS='0.4,0.7', RTHRESH='0.95', RMAX='2'),
}

## 4. 학습 실행 (끝날 때까지 이 셀이 돕니다. 약 80분. 1분마다 진행 상황 출력)

In [ ]:
import os, subprocess, time, json, glob
jobs = []
for name, cfg in EXPERIMENTS.items():
    out = f'{BASE}/{name}'; os.makedirs(out, exist_ok=True)
    for s in SEEDS:
        if os.path.exists(f'{out}/s{s}.json'):
            continue
        env = dict(os.environ, OUT=out, LISTENERS='1', REFRESH='0', THREADS='1',
                   DEVICE='cuda' if torch.cuda.is_available() else 'cpu', **cfg)
        jobs.append((name, s, env))
print('돌려야 할 학습:', len(jobs), '개')
running = []
def last_line(name, s):
    p = f'{BASE}/{name}/s{s}.log'
    if not os.path.exists(p): return '(시작 전)'
    L = [l for l in open(p).read().splitlines() if l.strip()]
    return L[-1][:110] if L else ''
t0 = time.time(); tick = 0
while jobs or running:
    running = [(n, s, p) for (n, s, p) in running if p.poll() is None]
    while jobs and len(running) < MAX_PARALLEL:
        n, s, env = jobs.pop(0)
        p = subprocess.Popen(['python', '-u', 'scripts/run_count_A.py', str(s), str(STEPS)], env=env,
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        running.append((n, s, p))
    if tick % 6 == 0:
        print(f'--- {int((time.time()-t0)/60)}분 경과, 실행 중 {len(running)}개, 대기 {len(jobs)}개')
        for n, s, _ in running:
            print(f'  {n} 시드{s}:', last_line(n, s))
    tick += 1
    time.sleep(10)
print('모두 끝났습니다')

## 5. 결과 요약

In [ ]:
import json, glob, numpy as np
def summarize():
    for name in EXPERIMENTS:
        print(f'===== {name}')
        accs, perfect = [], 0
        for s in SEEDS:
            p = f'{BASE}/{name}/s{s}.json'
            if not os.path.exists(p):
                print(f'  시드{s}: 아직'); continue
            d = json.load(open(p)); r = d['report']; e5 = r['e5']; adj = r['e4_adjacent']
            ok = sum(v >= 0.9 for v in adj.values())
            merged = {}
            for k, v in e5['modal'].items(): merged.setdefault(tuple(v), []).append(int(k))
            merged = [g for g in merged.values() if len(g) > 1]
            good = (r['e1_train'] >= 0.95 and ok == 9 and e5['consistency'] >= 0.95 and e5['injective'] >= 0.95)
            perfect += good; accs.append(r['e1_train'])
            print(f"  시드{s}: 정확도 {r['e1_train']:.3f} | 빼둔쌍 {r['e1_heldout_pairs']:.3f} | n대n+1(균형) ≥90% {ok}/9 | "
                  f"일관성 {e5['consistency']:.2f} 단사성 {e5['injective']:.2f} | 합쳐진 개수 {merged} | 재시작 {d.get('restarts', '?')}회 {d.get('restart_log', '')} | {'완벽' if good else ''}")
        if accs:
            print(f'  >> 끝난 시드 {len(accs)}개 중 완벽한 코드 {perfect}개 ({perfect/len(accs):.0%}), 정확도 평균 {np.mean(accs):.3f} ± {np.std(accs):.3f}')
            print('  (완벽 = 정확도≥0.95, 모든 n대n+1(균형) ≥90%, 일관성·단사성 ≥0.95)')
summarize()

## 6. (선택) 결과를 저장소에 올리기
`GITHUB_TOKEN` (쓰기 권한)이 Secrets 에 있을 때만 됩니다. 아니면 5번 셀 출력을 대화에 붙여주세요.

In [ ]:
import shutil, subprocess
assert token, 'GITHUB_TOKEN 이 없습니다 (Secrets 에 추가하세요)'
dst = '/content/integral_discovery/runs3/colab_A8'
for name in EXPERIMENTS:
    for f in glob.glob(f'{BASE}/{name}/s*.json') + glob.glob(f'{BASE}/{name}/s*.log'):
        os.makedirs(f'{dst}/{name}', exist_ok=True); shutil.copy(f, f'{dst}/{name}/')
cmds = [['git','config','user.email','colab@example.com'], ['git','config','user.name','colab'],
        ['git','add','runs3/colab_A8'], ['git','commit','-m','코랩 단계 A8(재시작) 결과'],
        ['git','pull','--no-edit','origin',BRANCH], ['git','push','origin',BRANCH]]
for c in cmds:
    r = subprocess.run(c, cwd='/content/integral_discovery', capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-300:])